In [9]:
import kagglehub
kaggle_path = kagglehub.dataset_download("itachi9604/disease-symptom-description-dataset")

Using Colab cache for faster access to the 'disease-symptom-description-dataset' dataset.


In [ ]:
import re
import os
import copy
import random

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Selected device: {device}")

PyTorch version: 2.11.0+cu130
Selected device: cuda


In [ ]:
df = pd.read_csv(os.path.join(kaggle_path, "dataset.csv"))

symptom_columns = [
    col for col in df.columns
    if col.lower().startswith("symptom")
]

if not symptom_columns:
    raise ValueError("No symptom columns were found.")

df = df.dropna(subset=["Disease"]).copy()

df["Disease"] = df["Disease"].astype(str).str.strip()

df = df[df["Disease"] != ""].copy()

print(f"Number of samples: {len(df)}")
print(f"Number of symptom columns: {len(symptom_columns)}")

df.head()

Number of samples: 4920
Number of symptom columns: 17


,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,Fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Fungal infection,itching,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Fungal infection,itching,skin_rash,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Fungal infection,itching,skin_rash,nodal_skin_eruptions,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
def normalize_symptom(symptom):
    """
    Normalize a symptom name.

    Examples:
        'continuous_sneezing' -> 'continuous sneezing'
        ' Continuous Sneezing ' -> 'continuous sneezing'
    """
    if pd.isna(symptom):
        return None

    symptom = str(symptom).strip().lower()

    if not symptom or symptom == "nan":
        return None
    
    symptom = symptom.replace("_", " ")
    symptom = re.sub(r"\s+", " ", symptom)

    return symptom

In [ ]:
def extract_symptoms(row):
    """ Extract unique, normalized symptoms from one sample. """
    symptoms = [
        normalize_symptom(value)
        for value in row[symptom_columns]
    ]

    symptoms = {
        symptom
        for symptom in symptoms
        if symptom is not None
    }

    return sorted(symptoms)


df["Symptoms"] = df.apply(extract_symptoms, axis=1)

df = df[["Disease", "Symptoms"]].reset_index(drop=True)

df.head()

,Disease,Symptoms
0,Fungal infection,"[dischromic patches, itching, nodal skin erupt..."
1,Fungal infection,"[dischromic patches, nodal skin eruptions, ski..."
2,Fungal infection,"[dischromic patches, itching, nodal skin erupt..."
3,Fungal infection,"[dischromic patches, itching, skin rash]"
4,Fungal infection,"[itching, nodal skin eruptions, skin rash]"


In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.25,
    random_state=SEED,
    stratify=df["Disease"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=SEED,
    stratify=temp_df["Disease"]
)

print(f"Training samples:   {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples:       {len(test_df)}")

Training samples:   3690
Validation samples: 615
Test samples:       615


In [ ]:
symptom_encoder = MultiLabelBinarizer()

X_train = symptom_encoder.fit_transform(
    train_df["Symptoms"]
)

X_val = symptom_encoder.transform(
    val_df["Symptoms"]
)

X_test = symptom_encoder.transform(
    test_df["Symptoms"]
)

feature_names = symptom_encoder.classes_

X_train = pd.DataFrame(
    X_train,
    columns=feature_names,
    index=train_df.index
)

X_val = pd.DataFrame(
    X_val,
    columns=feature_names,
    index=val_df.index
)

X_test = pd.DataFrame(
    X_test,
    columns=feature_names,
    index=test_df.index
)

print("Training features:", X_train.shape)
print("Validation features:", X_val.shape)
print("Test features:", X_test.shape)

Training features: (3690, 131)
Validation features: (615, 131)
Test features: (615, 131)


In [ ]:
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(
    train_df["Disease"]
)

y_val = label_encoder.transform(
    val_df["Disease"]
)

y_test = label_encoder.transform(
    test_df["Disease"]
)

num_classes = len(label_encoder.classes_)

print("Number of disease classes:", num_classes)
print("Disease classes:", label_encoder.classes_)

Number of disease classes: 41
Disease classes: ['(vertigo) Paroymsal  Positional Vertigo' 'AIDS' 'Acne'
 'Alcoholic hepatitis' 'Allergy' 'Arthritis' 'Bronchial Asthma'
 'Cervical spondylosis' 'Chicken pox' 'Chronic cholestasis' 'Common Cold'
 'Dengue' 'Diabetes' 'Dimorphic hemmorhoids(piles)' 'Drug Reaction'
 'Fungal infection' 'GERD' 'Gastroenteritis' 'Heart attack' 'Hepatitis B'
 'Hepatitis C' 'Hepatitis D' 'Hepatitis E' 'Hypertension'
 'Hyperthyroidism' 'Hypoglycemia' 'Hypothyroidism' 'Impetigo' 'Jaundice'
 'Malaria' 'Migraine' 'Osteoarthristis' 'Paralysis (brain hemorrhage)'
 'Peptic ulcer diseae' 'Pneumonia' 'Psoriasis' 'Tuberculosis' 'Typhoid'
 'Urinary tract infection' 'Varicose veins' 'hepatitis A']


In [ ]:
X_train_tensor = torch.tensor(
    X_train.to_numpy(dtype=np.float32),
    dtype=torch.float32
)

X_val_tensor = torch.tensor(
    X_val.to_numpy(dtype=np.float32),
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test.to_numpy(dtype=np.float32),
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long
)

y_val_tensor = torch.tensor(
    y_val,
    dtype=torch.long
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.long
)

print("X_train:", X_train_tensor.shape)
print("y_train:", y_train_tensor.shape)

print("X_val:", X_val_tensor.shape)
print("y_val:", y_val_tensor.shape)

print("X_test:", X_test_tensor.shape)
print("y_test:", y_test_tensor.shape)

X_train: torch.Size([3690, 131])
y_train: torch.Size([3690])
X_val: torch.Size([615, 131])
y_val: torch.Size([615])
X_test: torch.Size([615, 131])
y_test: torch.Size([615])


In [ ]:
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
class DiseaseClassifier(nn.Module):

    def __init__(self, input_size, num_classes):
        super().__init__()

        self.network = nn.Sequential(
            
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
input_size = X_train_tensor.shape[1]

model = DiseaseClassifier(
    input_size=input_size,
    num_classes=num_classes
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print(model)

DiseaseClassifier(
  (network): Sequential(
    (0): Linear(in_features=131, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=128, out_features=64, bias=True)
    (7): ReLU()
    (8): Linear(in_features=64, out_features=41, bias=True)
  )
)


In [ ]:
def evaluate_model(model, data_loader, criterion, device):
    """
    Evaluate the model without updating its parameters.

    Returns:
        Average loss and accuracy.
    """
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for X_batch, y_batch in data_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)

            loss = criterion(outputs, y_batch)

            predictions = outputs.argmax(dim=1)

            batch_size = y_batch.size(0)

            total_loss += loss.item() * batch_size
            total_correct += (
                predictions == y_batch
            ).sum().item()

            total_samples += batch_size

    average_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    return average_loss, accuracy

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs=100, patience=10):

    best_val_loss = float("inf")
    best_model_state = copy.deepcopy(model.state_dict())

    epochs_without_improvement = 0

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": []
    }

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0
        total_correct = 0
        total_samples = 0

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad(set_to_none=True)

            outputs = model(X_batch)

            loss = criterion(outputs, y_batch)

            loss.backward()

            optimizer.step()

            batch_size = y_batch.size(0)

            running_loss += loss.item() * batch_size

            predictions = outputs.argmax(dim=1)

            total_correct += (
                predictions == y_batch
            ).sum().item()

            total_samples += batch_size

        train_loss = running_loss / total_samples
        train_accuracy = total_correct / total_samples

        val_loss, val_accuracy = evaluate_model(
            model,
            val_loader,
            criterion,
            device
        )

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)

        print(
            f"Epoch [{epoch + 1:03d}/{epochs}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_accuracy:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_accuracy:.4f}"
        )

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            best_model_state = copy.deepcopy(
                model.state_dict()
            )

            epochs_without_improvement = 0

        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:

            print(
                f"\nEarly stopping triggered at "
                f"epoch {epoch + 1}."
            )

            break

    model.load_state_dict(best_model_state)

    print(f"\nBest validation loss: {best_val_loss:.4f}")

    return model, history

In [24]:
EPOCHS = 100
PATIENCE = 10

model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS,
    patience=PATIENCE
)

Epoch [001/100] | Train Loss: 2.1950 | Train Acc: 0.5453 | Val Loss: 0.1735 | Val Acc: 0.9967
Epoch [002/100] | Train Loss: 0.0794 | Train Acc: 0.9946 | Val Loss: 0.0090 | Val Acc: 1.0000
Epoch [003/100] | Train Loss: 0.0153 | Train Acc: 0.9989 | Val Loss: 0.0030 | Val Acc: 1.0000
Epoch [004/100] | Train Loss: 0.0085 | Train Acc: 0.9989 | Val Loss: 0.0013 | Val Acc: 1.0000
Epoch [005/100] | Train Loss: 0.0049 | Train Acc: 1.0000 | Val Loss: 0.0026 | Val Acc: 1.0000
Epoch [006/100] | Train Loss: 0.0051 | Train Acc: 0.9989 | Val Loss: 0.0005 | Val Acc: 1.0000
Epoch [007/100] | Train Loss: 0.0045 | Train Acc: 0.9995 | Val Loss: 0.0004 | Val Acc: 1.0000
Epoch [008/100] | Train Loss: 0.0020 | Train Acc: 1.0000 | Val Loss: 0.0002 | Val Acc: 1.0000
Epoch [009/100] | Train Loss: 0.0015 | Train Acc: 1.0000 | Val Loss: 0.0001 | Val Acc: 1.0000
Epoch [010/100] | Train Loss: 0.0014 | Train Acc: 1.0000 | Val Loss: 0.0001 | Val Acc: 1.0000
Epoch [011/100] | Train Loss: 0.0009 | Train Acc: 1.0000 | V

In [25]:
history_df = pd.DataFrame(history)

history_df.head()

,train_loss,train_accuracy,val_loss,val_accuracy
0,2.195010,0.545257,0.173509,0.996748
1,0.079384,0.994580,0.009024,1.000000
2,0.015300,0.998916,0.003007,1.000000
3,0.008451,0.998916,0.001265,1.000000
4,0.004934,1.000000,0.002562,1.000000


In [26]:
best_epoch = history_df["val_loss"].idxmin() + 1

print(f"Best epoch: {best_epoch}")
print(f"Best validation loss: {history_df['val_loss'].min():.4f}")
print(f"Validation accuracy at best epoch: "
      f"{history_df.loc[best_epoch - 1, 'val_accuracy']:.4f}")

Best epoch: 60
Best validation loss: 0.0000
Validation accuracy at best epoch: 1.0000


In [27]:
test_loss, test_accuracy = evaluate_model(
    model=model,
    data_loader=test_loader,
    criterion=criterion,
    device=device
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

Test Loss: 0.0000
Test Accuracy: 1.0000


In [28]:
model.eval()

all_predictions = []
all_targets = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        predictions = outputs.argmax(dim=1)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_targets.extend(
            y_batch.numpy()
        )

print(
    classification_report(
        all_targets,
        all_predictions,
        labels=np.arange(num_classes),
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

                                         precision    recall  f1-score   support

(vertigo) Paroymsal  Positional Vertigo       1.00      1.00      1.00        15
                                   AIDS       1.00      1.00      1.00        15
                                   Acne       1.00      1.00      1.00        15
                    Alcoholic hepatitis       1.00      1.00      1.00        15
                                Allergy       1.00      1.00      1.00        15
                              Arthritis       1.00      1.00      1.00        15
                       Bronchial Asthma       1.00      1.00      1.00        15
                   Cervical spondylosis       1.00      1.00      1.00        15
                            Chicken pox       1.00      1.00      1.00        15
                    Chronic cholestasis       1.00      1.00      1.00        15
                            Common Cold       1.00      1.00      1.00        15
                           

In [ ]:
def predict_disease(symptoms, top_k=3):
    """
    Predict disease classes from a list of symptoms.

    Args:
        symptoms: List of symptom names.
        top_k: Number of top predictions to return.

    Returns:
        A DataFrame containing predicted disease classes
        and their estimated probabilities.
    """

    if not symptoms:
        raise ValueError("Please provide at least one symptom.")

    cleaned_symptoms = [
        normalize_symptom(symptom)
        for symptom in symptoms
    ]

    cleaned_symptoms = list(
        dict.fromkeys(
            symptom
            for symptom in cleaned_symptoms
            if symptom is not None
        )
    )

    if not cleaned_symptoms:
        raise ValueError("No valid symptoms were provided.")

    known_symptoms = set(symptom_encoder.classes_)

    unknown_symptoms = [
        symptom
        for symptom in cleaned_symptoms
        if symptom not in known_symptoms
    ]

    if unknown_symptoms:
        print(
            "Warning: These symptoms were not found in the "
            f"training vocabulary: {unknown_symptoms}"
        )

    encoded_input = symptom_encoder.transform(
        [cleaned_symptoms]
    )

    user_tensor = torch.tensor(
        encoded_input,
        dtype=torch.float32,
        device=device
    )

    model.eval()

    with torch.no_grad():

        logits = model(user_tensor)

        probabilities = torch.softmax(
            logits,
            dim=1
        )[0]

    top_k = min(
        max(1, top_k),
        num_classes
    )

    top_probabilities, top_indices = torch.topk(
        probabilities,
        k=top_k
    )

    predicted_diseases = label_encoder.inverse_transform(
        top_indices.cpu().numpy()
    )

    results = pd.DataFrame({
        "Disease": predicted_diseases,
        "Probability": top_probabilities.cpu().numpy()
    })

    return results

In [30]:
user_input = [
    "stomach_pain",
    "acidity",
    "chest_pain"
]

predictions = predict_disease(
    user_input,
    top_k=3
)

predictions

,Disease,Probability
0,GERD,0.999913
1,Hypertension,0.000056
2,Psoriasis,0.000017


In [31]:
user_input = [
    "continuous_sneezing",
    "watering_from_eyes"
]

predictions = predict_disease(
    user_input,
    top_k=3
)

predictions

,Disease,Probability
0,Allergy,0.999976
1,(vertigo) Paroymsal Positional Vertigo,0.000009
2,Psoriasis,0.000008


In [ ]:
from pathlib import Path

model_dir = Path("models")
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "disease_classifier.pth"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "input_size": input_size,
        "num_classes": num_classes,
        "symptom_classes": symptom_encoder.classes_.tolist(),
        "disease_classes": label_encoder.classes_.tolist()
    },
    model_path
)

print(f"Model saved to: {model_path}")

Model saved to: models/disease_classifier.pth


In [ ]:
from pathlib import Path
import numpy as np
import torch
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder

model_path = Path("models/disease_classifier.pth")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(
    model_path,
    map_location=device,
    weights_only=True
)

model = DiseaseClassifier(
    input_size=checkpoint["input_size"],
    num_classes=checkpoint["num_classes"]
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

symptom_encoder = MultiLabelBinarizer(
    classes=checkpoint["symptom_classes"]
)

symptom_encoder.fit([[]])

label_encoder = LabelEncoder()
label_encoder.classes_ = np.array(checkpoint["disease_classes"])

print(f"Model loaded successfully on {device}.")
print(f"Number of symptoms: {len(symptom_encoder.classes_)}")
print(f"Number of diseases: {len(label_encoder.classes_)}")

Model loaded successfully on cuda.
Number of symptoms: 131
Number of diseases: 41


In [36]:
predictions = predict_disease(
    ["stomach_pain", "acidity", "chest_pain"],
    top_k=3
)

predictions

,Disease,Probability
0,GERD,0.999913
1,Hypertension,0.000056
2,Psoriasis,0.000017
